In [ ]:
# Import SymPy
from sympy import symbols, Matrix, conjugate, I, simplify, latex
from sympy.printing.latex import LatexPrinter
from sympy.core.mul import Mul
from sympy.functions.elementary.complexes import conjugate as Conjugate

# Custom LaTeX printer that:
# 1. Converts z*conjugate(z) to |z|^2
# 2. Converts \overline{variable} to variable^*
class CustomLaTeXPrinter(LatexPrinter):
    """Custom LaTeX printer with magnitude squared and asterisk conjugation notation"""
    
    def _print_Conjugate(self, expr):
        """Override conjugate printing to use asterisk superscript instead of overline"""
        # Get the LaTeX representation of the argument
        arg_latex = self._print(expr.args[0])
        # Return with asterisk superscript
        return arg_latex + r"^{*}"
    
    def _print_conjugate(self, expr):
        """Override conjugate printing (lowercase method name)"""
        return self._print_Conjugate(expr)
    
    def _print_Function(self, expr):
        """Override function printing to handle conjugate"""
        # Check if this is a conjugate function
        if expr.func.__name__ == 'conjugate' or str(expr.func) == 'conjugate':
            return self._print_Conjugate(expr)
        # Otherwise use default printing
        return super()._print_Function(expr)
    
    def _print_Mul(self, expr):
        """Override multiplication printing to detect conjugate pairs"""
        from sympy import Mul, Pow
        
        # Get the factors
        args = list(expr.args)
        
        # Separate numeric coefficients from symbolic factors
        coeffs = []
        symbolic_args = []
        for arg in args:
            if arg.is_number:
                coeffs.append(arg)
            else:
                symbolic_args.append(arg)
        
        # Check if any pair of symbolic args is a variable and its conjugate
        magnitude_base = None
        remaining_args = list(symbolic_args)
        
        for i, arg1 in enumerate(symbolic_args):
            for j, arg2 in enumerate(symbolic_args):
                if i < j:  # Only check each pair once
                    # Check if arg1 is conjugate(arg2)
                    if isinstance(arg1, Conjugate) and arg1.args[0] == arg2:
                        magnitude_base = arg2
                        remaining_args.remove(arg1)
                        remaining_args.remove(arg2)
                        break
                    # Check if arg2 is conjugate(arg1)
                    elif isinstance(arg2, Conjugate) and arg2.args[0] == arg1:
                        magnitude_base = arg1
                        remaining_args.remove(arg1)
                        remaining_args.remove(arg2)
                        break
            if magnitude_base is not None:
                break
        
        # If we found a magnitude pair
        if magnitude_base is not None:
            # Get LaTeX representation of the base
            base_latex = self._print(magnitude_base)
            magnitude_latex = r"\left|" + base_latex + r"\right|^{2}"
            
            # Build the result
            parts = []
            
            # Add coefficient if present
            if coeffs:
                coeff = Mul(*coeffs, evaluate=False)
                parts.append(self._print(coeff))
            
            # Add remaining factors if present - use parent's method to preserve structure
            # but it will still use our _print_Conjugate via self._print()
            if remaining_args:
                if len(remaining_args) == 1:
                    parts.append(self._print(remaining_args[0]))
                else:
                    remaining_mul = Mul(*remaining_args, evaluate=False)
                    # Use parent's method which handles structure correctly
                    # It will still call self._print() for conjugates
                    parts.append(super()._print_Mul(remaining_mul))
            
            # Add magnitude
            parts.append(magnitude_latex)
            
            return " ".join(parts)
        
        # If no conjugate pair found, use parent's method which handles divisions correctly
        # The _print_Conjugate override will still be called for any conjugates
        return super()._print_Mul(expr)

def latex_custom(expr, **settings):
    """LaTeX printer that converts z*conjugate(z) to |z|^2 and \overline{v} to v^*"""
    return CustomLaTeXPrinter(settings).doprint(expr)

def latex_matrix_factored(matrix, printer_func=latex_custom, **settings):
    """
    Convert a matrix to LaTeX, factoring out common denominators or factors.
    
    Parameters:
    -----------
    matrix : sympy.Matrix
        The matrix to convert
    printer_func : callable
        Function to use for LaTeX conversion (default: latex_custom)
    **settings
        Additional settings to pass to the printer
    
    Returns:
    --------
    str : LaTeX string with common factors factored out
    """
    from sympy import Mul, Pow, Integer, Rational, cancel, factor, Matrix
    
    # Get matrix dimensions
    rows, cols = matrix.shape
    
    # Extract all elements
    elements = [matrix[i, j] for i in range(rows) for j in range(cols)]
    
    # Find common denominator by looking at the structure of each element
    # SymPy represents fractions as Mul(numerator, Pow(denominator, -1))
    denominators = []
    numerators = []
    
    for elem in elements:
        # Check if element is a fraction (Mul with Pow with negative exponent)
        if isinstance(elem, Mul):
            denom_parts = []
            num_parts = []
            for arg in elem.args:
                if isinstance(arg, Pow) and arg.exp.is_negative:
                    # This is 1/denominator, so denominator is arg.base
                    denom_parts.append(arg.base)
                else:
                    # This is part of the numerator
                    num_parts.append(arg)
            
            if denom_parts:
                # Combine all denominator parts
                if len(denom_parts) == 1:
                    denominators.append(denom_parts[0])
                else:
                    denominators.append(Mul(*denom_parts))
                
                # Combine numerator parts
                if num_parts:
                    numerators.append(Mul(*num_parts))
                else:
                    numerators.append(Integer(1))
            else:
                # No denominator, element is not a fraction
                denominators.append(Integer(1))
                numerators.append(elem)
        else:
            # Not a Mul, so not a fraction
            denominators.append(Integer(1))
            numerators.append(elem)
    
    # Find common denominator (if all denominators are the same)
    # Check if all denominators are equal (using simplify to handle equivalent forms)
    from sympy import simplify
    first_denom = denominators[0]
    all_same = True
    for denom in denominators[1:]:
        if simplify(first_denom - denom) != Integer(0):
            all_same = False
            break
    
    if all_same and first_denom != Integer(1):
        # Don't factor out denominator for 1x1 matrices
        if rows == 1 and cols == 1:
            # For 1x1, just return the regular LaTeX
            return printer_func(matrix, **settings)
        
        common_denom = first_denom
        # Create new matrix with numerators only
        factored_matrix = Matrix(rows, cols, lambda i, j: numerators[i*cols + j])
        
        # Generate LaTeX
        matrix_latex = printer_func(factored_matrix, **settings)
        denom_latex = printer_func(common_denom, **settings)
        
        # Return as (1/denom) * matrix
        return r"\frac{1}{" + denom_latex + r"} " + matrix_latex
    
    # Try to find common factors in numerators
    # This is trickier - we'll look for factors that appear in all non-zero elements
    from sympy import gcd, lcm
    
    # Get all non-zero elements
    non_zero = [n for n in numerators if n != Integer(0)]
    
    if len(non_zero) > 1:
        # Try to find common factors using gcd
        try:
            # Convert to a list we can work with
            common_factor = non_zero[0]
            for n in non_zero[1:]:
                # Try to find common factors
                # This is approximate - we'll look for structural similarity
                if isinstance(common_factor, Mul) and isinstance(n, Mul):
                    # Compare factor structures
                    common_factor = gcd(common_factor, n)
                else:
                    common_factor = gcd(common_factor, n)
            
            # If we found a meaningful common factor (not just 1)
            if common_factor != Integer(1) and common_factor != Integer(-1):
                # Check if all elements are divisible by this factor
                factored_elements = []
                all_divisible = True
                for i, n in enumerate(numerators):
                    if n == Integer(0):
                        factored_elements.append(Integer(0))
                    else:
                        try:
                            factored = cancel(n / common_factor)
                            factored_elements.append(factored)
                        except:
                            all_divisible = False
                            break
                
                if all_divisible:
                    factored_matrix = Matrix(rows, cols, lambda i, j: factored_elements[i*cols + j])
                    matrix_latex = printer_func(factored_matrix, **settings)
                    factor_latex = printer_func(common_factor, **settings)
                    return factor_latex + r" " + matrix_latex
        except:
            pass
    
    # No common factors found, return regular matrix
    return printer_func(matrix, **settings)

# Define symbols
Delta_A, Delta_B, Delta_C, Delta_D, beta_BA, beta_CA, beta_DB, beta_DC = symbols('Delta_A Delta_B Delta_C Delta_D beta_BA beta_CA beta_DB beta_DC', complex=True)

# Build matrix M
M = Matrix([
    [Delta_A, beta_BA, beta_CA, 0],
    [conjugate(beta_BA), Delta_B, 0, beta_DB],
    [conjugate(beta_CA), 0, Delta_C, beta_DC],
    [0, conjugate(beta_DB), conjugate(beta_DC), Delta_D]
])

# Display matrix
print(M)

# ======================================================================
# Kron Reduction
# ======================================================================

# Using the matrix M defined above
# Nodes to keep: ['A', 'B']
# Nodes to eliminate: ['C', 'D']

# Partition M into blocks:
# M = [M_AA  M_AB]
#     [M_BA  M_BB]
# where A = kept nodes, B = eliminated nodes

kept_indices = [0, 1]
elim_indices = [2, 3]

# Extract blocks
M_AA = Matrix([[M[i, j] for j in kept_indices] for i in kept_indices])
M_AB = Matrix([[M[i, j] for j in elim_indices] for i in kept_indices])
M_BA = Matrix([[M[i, j] for j in kept_indices] for i in elim_indices])
M_BB = Matrix([[M[i, j] for j in elim_indices] for i in elim_indices])

# Compute Schur complement (Kron-reduced matrix)
M_kron = M_AA - M_AB * M_BB.inv() * M_BA

# Simplify
M_kron = simplify(M_kron)

# Display
print("Kron-reduced matrix M_kron:")
print(M_kron)
print("\nLaTeX (standard):")
print(latex(M_kron))
print("\nLaTeX (custom: magnitude squared + asterisk conjugation):")
print(latex_custom(M_kron))
print("\nLaTeX (factored - common denominator extracted):")
print(latex_matrix_factored(M_kron))


<>:105: SyntaxWarning: invalid escape sequence '\o'
<>:105: SyntaxWarning: invalid escape sequence '\o'
/var/folders/vg/y4r4cf451mz1kvjy8py8y9640000gn/T/ipykernel_29121/1022510518.py:105: SyntaxWarning: invalid escape sequence '\o'
  """LaTeX printer that converts z*conjugate(z) to |z|^2 and \overline{v} to v^*"""


Matrix([[Delta_A, beta_BA, beta_CA, 0], [conjugate(beta_BA), Delta_B, 0, beta_DB], [conjugate(beta_CA), 0, Delta_C, beta_DC], [0, conjugate(beta_DB), conjugate(beta_DC), Delta_D]])
Kron-reduced matrix M_kron:
Matrix([[(Delta_A*(Delta_C*Delta_D - beta_DC*conjugate(beta_DC)) - Delta_D*beta_CA*conjugate(beta_CA))/(Delta_C*Delta_D - beta_DC*conjugate(beta_DC)), (beta_BA*(Delta_C*Delta_D - beta_DC*conjugate(beta_DC)) + beta_CA*beta_DC*conjugate(beta_DB))/(Delta_C*Delta_D - beta_DC*conjugate(beta_DC))], [(beta_DB*conjugate(beta_CA)*conjugate(beta_DC) + (Delta_C*Delta_D - beta_DC*conjugate(beta_DC))*conjugate(beta_BA))/(Delta_C*Delta_D - beta_DC*conjugate(beta_DC)), (Delta_B*(Delta_C*Delta_D - beta_DC*conjugate(beta_DC)) - Delta_C*beta_DB*conjugate(beta_DB))/(Delta_C*Delta_D - beta_DC*conjugate(beta_DC))]])

LaTeX (standard):
\left[\begin{matrix}\frac{\Delta_{A} \left(\Delta_{C} \Delta_{D} - \beta_{DC} \overline{\beta_{DC}}\right) - \Delta_{D} \beta_{CA} \overline{\beta_{CA}}}{\Delta_{C} \Del

In [2]:
# Test the custom printer with examples
print("=== Test 1: Magnitude squared conversion ===")
test_expr1 = beta_DC * conjugate(beta_DC)
print("Expression:", test_expr1)
print("Standard LaTeX:", latex(test_expr1))
print("Custom LaTeX:", latex_custom(test_expr1))

print("\n=== Test 2: Asterisk conjugation notation ===")
test_expr2 = conjugate(beta_BA)
print("Expression:", test_expr2)
print("Standard LaTeX:", latex(test_expr2))
print("Custom LaTeX:", latex_custom(test_expr2))

print("\n=== Test 3: Combined (conjugate in product) ===")
test_expr3 = beta_CA * conjugate(beta_DB)
print("Expression:", test_expr3)
print("Standard LaTeX:", latex(test_expr3))
print("Custom LaTeX:", latex_custom(test_expr3))


=== Test 1: Magnitude squared conversion ===
Expression: beta_DC*conjugate(beta_DC)
Standard LaTeX: \beta_{DC} \overline{\beta_{DC}}
Custom LaTeX: \left|\beta_{DC}\right|^{2}

=== Test 2: Asterisk conjugation notation ===
Expression: conjugate(beta_BA)
Standard LaTeX: \overline{\beta_{BA}}
Custom LaTeX: \beta_{BA}^{*}

=== Test 3: Combined (conjugate in product) ===
Expression: beta_CA*conjugate(beta_DB)
Standard LaTeX: \beta_{CA} \overline{\beta_{DB}}
Custom LaTeX: \beta_{CA} \beta_{DB}^{*}


In [3]:
from joeutils.gojoe import *

JOEPLOT loaded. Includes Numpy, Seaborn, and matplotlib.pyplot as np, sns, plt respectively.
Imported PI, HBAR, Q, KB, PHI0, VARPHI0 [mks], and VARPHI018 [uA*pH].


In [4]:
jprint('$'+latex(M_kron)+'$')

$\left[\begin{matrix}\frac{\Delta_{A} \left(\Delta_{C} \Delta_{D} - \beta_{DC} \overline{\beta_{DC}}\right) - \Delta_{D} \beta_{CA} \overline{\beta_{CA}}}{\Delta_{C} \Delta_{D} - \beta_{DC} \overline{\beta_{DC}}} & \frac{\beta_{BA} \left(\Delta_{C} \Delta_{D} - \beta_{DC} \overline{\beta_{DC}}\right) + \beta_{CA} \beta_{DC} \overline{\beta_{DB}}}{\Delta_{C} \Delta_{D} - \beta_{DC} \overline{\beta_{DC}}}\\\frac{\beta_{DB} \overline{\beta_{CA}} \overline{\beta_{DC}} + \left(\Delta_{C} \Delta_{D} - \beta_{DC} \overline{\beta_{DC}}\right) \overline{\beta_{BA}}}{\Delta_{C} \Delta_{D} - \beta_{DC} \overline{\beta_{DC}}} & \frac{\Delta_{B} \left(\Delta_{C} \Delta_{D} - \beta_{DC} \overline{\beta_{DC}}\right) - \Delta_{C} \beta_{DB} \overline{\beta_{DB}}}{\Delta_{C} \Delta_{D} - \beta_{DC} \overline{\beta_{DC}}}\end{matrix}\right]$

In [5]:
jprint('$'+latex_custom(M_kron)+'$')

$\left[\begin{matrix}\frac{\Delta_{A} \left(\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}\right) - \Delta_{D} \left|\beta_{CA}\right|^{2}}{\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}} & \frac{\beta_{BA} \left(\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}\right) + \beta_{CA} \beta_{DC} \beta_{DB}^{*}}{\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}}\\\frac{\beta_{DB} \beta_{CA}^{*} \beta_{DC}^{*} + \left(\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}\right) \beta_{BA}^{*}}{\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}} & \frac{\Delta_{B} \left(\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}\right) - \Delta_{C} \left|\beta_{DB}\right|^{2}}{\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}}\end{matrix}\right]$

In [6]:
jdisplay_matrix = lambda expression: jprint('$'+latex_matrix_factored(expression)+'$')

In [7]:
jdisplay_matrix(M_kron)

$\frac{1}{\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}} \left[\begin{matrix}\Delta_{A} \left(\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}\right) - \Delta_{D} \left|\beta_{CA}\right|^{2} & \beta_{BA} \left(\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}\right) + \beta_{CA} \beta_{DC} \beta_{DB}^{*}\\\beta_{DB} \beta_{CA}^{*} \beta_{DC}^{*} + \left(\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}\right) \beta_{BA}^{*} & \Delta_{B} \left(\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}\right) - \Delta_{C} \left|\beta_{DB}\right|^{2}\end{matrix}\right]$

In [8]:
jdisplay_matrix(M)

$\left[\begin{matrix}\Delta_{A} & \beta_{BA} & \beta_{CA} & 0\\\beta_{BA}^{*} & \Delta_{B} & 0 & \beta_{DB}\\\beta_{CA}^{*} & 0 & \Delta_{C} & \beta_{DC}\\0 & \beta_{DB}^{*} & \beta_{DC}^{*} & \Delta_{D}\end{matrix}\right]$

In [11]:
print(latex_matrix_factored(M_kron))

\frac{1}{\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}} \left[\begin{matrix}\Delta_{A} \left(\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}\right) - \Delta_{D} \left|\beta_{CA}\right|^{2} & \beta_{BA} \left(\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}\right) + \beta_{CA} \beta_{DC} \beta_{DB}^{*}\\\beta_{DB} \beta_{CA}^{*} \beta_{DC}^{*} + \left(\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}\right) \beta_{BA}^{*} & \Delta_{B} \left(\Delta_{C} \Delta_{D} - \left|\beta_{DC}\right|^{2}\right) - \Delta_{C} \left|\beta_{DB}\right|^{2}\end{matrix}\right]
